# Research notebook · dancer floor positions

Goal: find a stable floor point for each visible dancer in this video. Cells analyze saved experiment evidence. The README has the separate commands to rerun inference scoring and training.

In [1]:
from pathlib import Path
from collections import Counter
import json
from IPython.display import Image, Video, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/build_cvat_packs.py").exists()), None)
assert ROOT is not None, "Start Jupyter from the project folder."
EXP = ROOT / "Experiment/Dance_Overnight"
def read(name):
    return json.loads((EXP / name).read_text(encoding="utf-8"))


### 1. Try a point derived from the box

A box bottom is cheap. We also tried correcting it with box geometry alone.

In [2]:
g = read("geometry/results.json")["chosen"]["validation"]["selected"]
v = read("ending_experiment/results.json")["reference"]["old_development"]["selected"]
for name, r in [("Box geometry", g), ("Image-based support", v)]:
    print(f"{name}: {r['correct']}/{r['refs']} within 3%; hidden reported visible: {r['hidden_emitted']}")


Box geometry: 88/104 within 3%; hidden reported visible: 29
Image-based support: 99/104 within 3%; hidden reported visible: 4


The image-based model placed more points correctly and produced fewer false visible contacts. We kept the detector for boxes and trained a separate head for point and visibility.

### 2. Test smoothing on the same tracks

In [3]:
r = read("round2/previous_test_diagnostic.json")["continuity"]
for name, x in r.items():
    print(f"{name}: p99 acceleration {x['acceleration_p99']:.3f}; abrupt changes {x['abrupt_changes_over_half_pct']}")


baseline: p99 acceleration 0.875; abrupt changes 90
candidate: p99 acceleration 0.049; abrupt changes 0


The offline fit reduced abrupt changes on this diagnostic replay. It uses future frames and can blur fast motion.

### 3. Try more data and heavier alternatives

In [4]:
m = read("data/development_ending.json")
for split in ("train", "val"):
    frames = [(d, k) for d in m.values() for k in d[split]]
    crops = [o for d, k in frames for o in d["labels"][str(k)] if o["box"] is not None and o["contact"] != "uncertain"]
    print(split, len(frames), "frames;", len(crops), "usable crops")

d = read("detector_expanded/tracking_selection.json")
for name in ("baseline", "chosen"):
    v = d[name]["validation"]
    print(f"Detector {name}: {v['correct_initial_ids']}/{v['reference_boxes']} matched the initial ID")
t = read("temporal_experiment/results.json")["runs"]
for name, r in t.items():
    print(name, r["controlled_development"]["correct_by_tolerance_pct_width"]["3"], "/ 77")
s = read("seed_repeatability/results.json")["models"]
for name, r in s.items():
    print(name, r["controlled"]["correct_by_tolerance_pct_width"]["3"], "/ 77")


train 60 frames; 969 usable crops
val 11 frames; 180 usable crops
Detector baseline: 100/120 matched the initial ID
Detector chosen: 101/120 matched the initial ID
v8_repeated_current_s17 72 / 77
v8_temporal2_s17 72 / 77
v8_temporal5_s17 72 / 77
v6_attention_wide_s17 74 / 77
v9_repeat_wide_s29 73 / 77
v9_repeat_wide_s43 73 / 77
ensemble 74 / 77


Extra posture examples stayed. The detector change gave a small gain in this ID check. Neighboring frames and averaging three models did not beat the selected single-image head on this development comparison.

### 4. Freeze the pipeline and score fresh frames

Count every visible reference; a missing prediction is a miss. Distances are percentages of mat width.

In [5]:
r = read("final_confirmation/results.json")
for name, model in r["candidates"].items():
    refs = [x for x in model["records"] if x["contact"] == "visible"]
    correct = sum(x["error_pct"] is not None and x["error_pct"] <= 3 for x in refs)
    hidden = [x for x in model["records"] if x["contact"] == "hidden"]
    false_visible = sum(x["state"] == 1 for x in hidden)
    assert correct == model["all"]["correct_by_tolerance_pct_width"]["3"]
    print(f"{name}: {correct}/{len(refs)} within 3%; {false_visible}/{len(hidden)} hidden marked visible")


Incumbent support: 25/46 within 3%; 8/50 hidden marked visible
Expanded support: 29/46 within 3%; 5/50 hidden marked visible
Selected pipeline / box bottom: 19/46 within 3%; 5/50 hidden marked visible


Keep **YOLO → tracking → learned support and visibility → offline smoothing**. The improvement comes from seven frames in one choreography with draft reference labels. Dense overlaps still lose positions and identities.

In [6]:
display(Video(url="Experiment/Dance_Overnight/videos/final_confirmation_point_ablation.mp4", width=950))


### 5. Try filling a few more gaps

In [7]:
d = read("weak_contact/results.json")["previous_confirmation_diagnostic"]
for name, r in d.items():
    a = r["all"]
    print(name, a["correct_by_tolerance_pct_width"]["3"], "/ 46; hidden estimates:", a["hidden_estimated"])


original 29 / 46; hidden estimates: 2
weak_estimates 31 / 46; hidden estimates: 8


The replay adds a few correct positions and more guesses for hidden dancers. Keep these hollow estimates optional; this is a replay of inspected frames.

Next: human review, retrain from the corrected labels, then write the separate end-to-end notebook once the final pipeline is settled.